# [데모 E] 퀴즈 형식 모음 — 11가지

> 개념 확인·코드 읽기·예측·리뷰를 노트북 안에서 해 보는 퀴즈 예시다. 문제 내용은 교안 슬라이드(1-3, 2-1, 4-1)와 실습 코드에서 가져왔다.

| # | 형식 | 도구 | 이럴 때 쓴다 |
|---|---|---|---|
| 1 | 객관식 + 코드 보기 | jupyterquiz | 코드 한 줄의 역할 확인 |
| 2 | 복수 선택 | jupyterquiz | "모두 고르시오" |
| 3 | 숫자 입력 | jupyterquiz | 토큰 수·청크 수처럼 답이 숫자인 것 |
| 4 | 글자 입력 | jupyterquiz | 메서드·용어 이름 |
| 5 | 문제 은행에서 무작위 출제 | jupyterquiz + JSON 파일 | 쉬는 시간 복습, 매번 다른 문제 |
| 6 | 플래시카드 | jupytercards | 비개발자 용어 복습 |
| 7 | 순서 맞추기 | 직접 만든 위젯 | 단계·흐름 순서 |
| 8 | 틀린 줄 찾기 | 직접 만든 위젯 | 코드 리뷰(Bug Hunt) |
| 9 | 예측하고 실행하기 | 노트북 셀 | 실행 결과를 먼저 추측하게 할 때 |
| 10 | 코드 빈칸 + 자동 채점 | ipytest | CodeMate에게 시킨 코드를 테스트로 확인 |
| 11 | 서술형 + LLM 채점 | 사내 LLM | "왜"를 묻는 문제, 4.2 평가 실습과 연결 |

**공통 한계:** 답은 각자 화면에서만 확인한다. 강사에게 모이지 않는다. 11번만 LLM을 호출한다.

In [ ]:
import html
from pathlib import Path

from IPython.display import HTML
from jupyterquiz import display_quiz

from assets.widgets import show_line_pick, show_order_quiz

## 1. 객관식 + 코드 보기 (1.3 Tool Calling)

보기에 코드 조각을 넣을 수 있다. 보기 순서는 매번 섞인다.

In [ ]:
display_quiz([{
    "question": "run_with_tools 안에서 도구 함수를 실제로 실행하는 줄은?",
    "type": "multiple_choice",
    "answer_cols": 1,
    "answers": [
        {"answer": "", "code": "response = llm_with_tools.invoke(messages)", "correct": False,
         "feedback": "모델을 부르는 줄이다. 모델은 tool_calls만 만든다."},
        {"answer": "", "code": "result = tool_map[call['name']].invoke(call['args'])", "correct": True,
         "feedback": "맞다. 앱이 이름으로 함수를 찾아 실행한다(1-3 p10)."},
        {"answer": "", "code": "llm_with_tools = llm.bind_tools(tools)", "correct": False,
         "feedback": "도구 스키마를 붙일 뿐, 실행하지 않는다."},
    ],
}], max_width=800)

## 2. 복수 선택 (4.1 하네스)

맞는 보기를 **모두** 골라야 정답이다.

In [ ]:
display_quiz([{
    "question": "하네스가 Agent에게 정해 주는 것을 모두 고르시오. (4-1 p9)",
    "type": "many_choice",
    "answers": [
        {"answer": "목표와 제약 조건", "correct": True, "feedback": "맞다."},
        {"answer": "쓸 수 있는 도구와 금지 행동", "correct": True, "feedback": "맞다."},
        {"answer": "중간·최종 결과를 검증할 기준", "correct": True, "feedback": "맞다."},
        {"answer": "실패했을 때 멈출지, 재시도할지, 사람에게 물을지", "correct": True, "feedback": "맞다."},
        {"answer": "모델 가중치를 다시 학습하는 방법", "correct": False, "feedback": "하네스는 모델을 바꾸지 않고 운영 구조를 설계한다."},
        {"answer": "최대한 긴 프롬프트", "correct": False, "feedback": "p8: 더 긴 지시문이 아니라 일이 끝나는 조건을 준다."},
    ],
}], max_width=800)

## 3. 숫자 입력 (1.2 토큰, 2.1 청킹)

**정답을 코드로 계산해 문제에 넣는다.** 문장이나 설정을 바꿔도 정답이 어긋나지 않는다. 정확한 값, 근처 범위(오답이지만 다른 해설), 그 밖의 값 순서로 채점한다.

In [ ]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

sentence = "LangGraph는 상태를 가진 에이전트를 만드는 라이브러리다."
tokens = len(tiktoken.get_encoding("o200k_base").encode(sentence))

doc = Path("../src/raw/Chapter02/06-capstone/data/ai_trends.md").read_text(encoding="utf-8")[:700]
chunks = len(RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0).split_text(doc))

display_quiz([
    {
        "question": f"다음 문장은 o200k_base 토크나이저로 몇 토큰인가? (글자 수는 {len(sentence)}자)",
        "code": sentence,
        "type": "numeric",
        "answers": [
            {"type": "value", "value": tokens, "correct": True, "feedback": f"맞다. {tokens}토큰이다. 글자 수와 토큰 수는 다르다."},
            {"type": "range", "range": [tokens - 4, tokens + 5], "correct": False, "feedback": "가깝다. 한국어는 한 글자가 한 토큰이 아닐 수 있다."},
            {"type": "default", "feedback": "4-1절 토큰 색칠 그림을 다시 보고 추측해 본다."},
        ],
    },
    {
        "question": "캡스톤 문서 앞 700자를 chunk_size=200, chunk_overlap=0으로 자르면 청크가 몇 개인가?",
        "type": "numeric",
        "answers": [
            {"type": "value", "value": chunks, "correct": True, "feedback": f"맞다. {chunks}개다. 분할기는 문단·줄바꿈에서 먼저 자르므로 700÷200보다 많을 수 있다."},
            {"type": "range", "range": [1, chunks], "correct": False, "feedback": "너무 적다. 700÷200으로 계산했다면, 분할기가 문단 경계에서 먼저 자른다는 점을 떠올린다."},
            {"type": "default", "feedback": "데모 D 4-2절의 청크 색칠 그림을 다시 본다."},
        ],
    },
], max_width=800)

## 4. 글자 입력 (2.2 LangGraph)

대소문자 무시, 오타 조금 허용(유사도 0.85 이상) 같은 채점 기준을 문제마다 정할 수 있다.

In [ ]:
display_quiz([
    {
        "question": "StateGraph에서 조건부 엣지를 추가하는 메서드 이름은?",
        "type": "string",
        "input_width": 24,
        "answers": [
            {"answer": "add_conditional_edges", "correct": True, "match_case": True, "feedback": "정확하다."},
            {"answer": "add_conditional_edges", "correct": True, "match_case": False, "fuzzy_threshold": 0.85,
             "feedback": "거의 맞다. 철자를 한 번 더 확인한다."},
            {"type": "default", "feedback": "데모 B 코드 지도의 '그래프 조립' 구획을 본다."},
        ],
    },
    {
        "question": "여러 노드가 같은 State 필드를 갱신할 때 값을 합치는 규칙을 영어로 무엇이라 하나?",
        "type": "string",
        "answers": [
            {"answer": "Reducer", "correct": True, "match_case": False, "feedback": "맞다(2-1 p10)."},
            {"answer": "Router", "correct": False, "match_case": False, "feedback": "Router는 다음 노드를 고르는 쪽이다."},
            {"type": "default", "feedback": "add_messages가 무엇의 예인지 떠올린다."},
        ],
    },
], max_width=800)

## 5. 문제 은행에서 무작위 출제

문제를 JSON 파일(`assets/quizzes/langgraph_bank.json`, 5문제)로 따로 두고, 셀을 실행할 때마다 **2문제를 무작위로** 뽑는다. 문제 수정은 JSON만 고치면 된다. CodeMate에게 "슬라이드 2-1로 문제 5개를 이 형식에 맞춰 더 만들어 줘"라고 시키기도 쉽다.

In [ ]:
display_quiz("assets/quizzes/langgraph_bank.json", num=2, shuffle_questions=True, max_width=800)

## 6. 플래시카드 (용어 복습)

카드를 클릭하면 뒤집힌다. 비개발자에게 어려운 용어를 쉬는 시간에 넘겨 보게 한다.

In [ ]:
from jupytercards import display_flashcards

display_flashcards([
    {"front": "Tool Schema", "back": "도구 이름·설명·인자 형식. 모델이 도구를 고를 때 본다."},
    {"front": "tool_calls", "back": "모델이 만든 '이 도구를 이 인자로 불러 달라'는 요청. 실행은 앱이 한다."},
    {"front": "Reducer", "back": "여러 노드가 같은 State 필드를 바꿀 때 값을 합치는 규칙."},
    {"front": "Checkpointer", "back": "그래프 실행 상태를 저장해 중단·재개를 가능하게 한다."},
    {"front": "Human-in-the-loop", "back": "되돌리기 어려운 행동 전에 사람의 승인을 받는 설계."},
    {"front": "SKILL.md", "back": "Agent가 특정 작업을 할 때 읽는 절차 문서. 언제·어떻게·검증 방법을 적는다."},
], title="AI Agent 용어")

## 7. 순서 맞추기 (직접 만든 위젯)

카드를 순서대로 클릭하고 채점한다. 틀린 칸에는 정답이 표시된다.

In [ ]:
show_order_quiz(
    ["목표 정의", "작업 분해", "도구 연결", "검증 설계", "예외 처리"],
    title="하네스 설계의 기본 순서 (4-1 p10)",
    explanation="완료 조건을 먼저 정해야 어디까지 할지 알 수 있고, 검증 기준이 있어야 실패를 판단할 수 있다.",
)

In [ ]:
show_order_quiz(
    ["Loading", "Splitting", "Embedding", "Storage", "Retrieval", "Generation"],
    title="RAG 핵심 모듈 순서 (2.1)",
    explanation="문서를 불러와 자르고, 벡터로 바꿔 저장한 뒤, 질문과 가까운 조각을 찾아 답을 만든다.",
)

## 8. 틀린 줄 찾기 (직접 만든 위젯)

데모 A의 수동 루프에 버그를 하나 넣었다. 문제가 있는 줄을 클릭한다.

In [ ]:
show_line_pick(
    """
def run_with_tools(question: str) -> str:
    messages = [HumanMessage(content=question)]
    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        if not response.tool_calls:
            return response.content
        for call in response.tool_calls:
            result = tool_map[call['name']].invoke(call['args'])
            messages.append(ToolMessage(content=str(result), tool_call_id=call['name']))
""",
    answer_lines=[10],
    explanation="tool_call_id에는 call['id']를 넣어야 한다. 이름을 넣으면 같은 도구를 두 번 불렀을 때 어느 결과가 어느 요청의 것인지 짝이 맞지 않는다.",
)

## 9. 예측하고 실행하기 (2.2 Reducer)

실행 **전에** 결과를 적는다. 그다음 실행해서 맞춰 본다. 예측이 틀린 지점이 가장 잘 기억된다.

`add_messages`에 메시지를 더하면 목록 길이는 몇이 될까? 아래 두 칸에 숫자를 적고 다음 셀을 실행한다.

In [ ]:
예측_1 = None   # 서로 다른 id의 메시지 1개 + 1개를 합치면?
예측_2 = None   # 같은 id의 메시지를 다시 넣으면?

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph.message import add_messages

기존 = [HumanMessage(content="안녕", id="m1")]
실제_1 = add_messages(기존, [AIMessage(content="반가워", id="m2")])
실제_2 = add_messages(기존, [HumanMessage(content="수정된 안녕", id="m1")])


def 채점(예측, 실제, 설명):
    if 예측 is None:
        print(f"먼저 예측을 적는다. (정답: {len(실제)})")
    else:
        print(("✔ 맞다" if 예측 == len(실제) else f"✖ 틀렸다 (정답: {len(실제)})") + f" — {설명}")


채점(예측_1, 실제_1, "id가 다르면 뒤에 누적된다.")
채점(예측_2, 실제_2, f"id가 같으면 교체된다. 첫 메시지 내용: {실제_2[0].content!r}")

## 10. 코드 빈칸 + 자동 채점 (ipytest)

빈칸은 수강생이 직접 치지 않고 **CodeMate에게 시킨다.** 채점은 테스트가 한다. 처음에는 빈칸이 비어 있어 테스트가 실패하는 것이 정상이다.

> **CodeMate 지시문:** 아래 셀의 `tool_map`을 채워 줘. 도구 이름으로 도구 객체를 찾을 수 있는 dict여야 하고, `tools` 목록이 바뀌어도 따로 고칠 필요가 없게 만들어 줘.

In [ ]:
import ipytest
from langchain_core.tools import tool

ipytest.autoconfig()


@tool
def get_weather(city: str) -> str:
    """지정한 도시의 현재 날씨를 조회한다."""
    return {"서울": "맑음, 기온 22°C", "부산": "흐림, 기온 24°C"}.get(city, f"{city}의 날씨 데이터를 찾을 수 없습니다.")


@tool
def calculate(expression: str) -> str:
    """수학 수식을 계산한다. 예: '2 + 3 * 4'"""
    return str(eval(expression, {"__builtins__": {}}))  # noqa: S307


tools = [get_weather, calculate]

tool_map = {}   # ← 빈칸: CodeMate에게 채우게 한다

In [ ]:
%%ipytest -q --tb=line


def test_이름으로_도구를_찾는다():
    assert tool_map["get_weather"].invoke({"city": "서울"}) == "맑음, 기온 22°C"


def test_모든_도구가_들어_있다():
    assert set(tool_map) == {t.name for t in tools}

**볼 지점:** 지금은 두 테스트가 모두 실패한다. 빈칸을 채운 뒤 두 셀을 다시 실행해 모두 통과하는지 본다.
CodeMate가 `{"get_weather": get_weather, "calculate": calculate}`처럼 손으로 적어 주면 테스트는 통과하지만 지시문의 두 번째 조건("목록이 바뀌어도")은 어긴다. **테스트 통과와 리뷰는 다르다**는 점을 짚는다.

## 11. 서술형 + LLM 채점 (LLM 호출)

"왜"를 묻는 문제는 보기로 만들기 어렵다. 수강생이 쓴 답을 LLM이 **채점 기준**으로 평가하게 한다. 채점 기준을 글로 적는 일 자체가 4.2 평가 실습(LLM-as-a-judge)의 연습이다.

**한계:** LLM 채점도 틀릴 수 있다. 점수보다 근거를 읽고, 이상하면 강사가 판단한다.

In [ ]:
from pydantic import BaseModel, Field

from config.llm import get_llm

QUESTION = "Tool Calling에서 모델이 도구 함수를 직접 실행하지 않고 tool_calls만 만드는 이유는 무엇인가?"
RUBRIC = """
- 모델은 텍스트(구조화된 호출 의도)를 생성할 뿐, 코드를 실행할 수 없다는 점을 말한다. (1점)
- 실제 실행은 애플리케이션(런타임)이 맡는다는 점을 말한다. (1점)
- 그 결과 실행 권한·안전·검증을 앱이 통제할 수 있다는 장점을 말한다. (1점)
"""


class Grade(BaseModel):
    점수: int = Field(description="0~3점")
    근거: str = Field(description="채점 기준 각 항목을 충족했는지 한 줄씩")
    보완할_점: str = Field(description="답을 더 좋게 만들 한 문장")


grader = get_llm().with_structured_output(Grade)


def grade(answer: str) -> HTML:
    g = grader.invoke(f"문제: {QUESTION}\n채점 기준:\n{RUBRIC}\n학생 답:\n{answer}\n\n채점 기준만으로 채점하라.")
    return HTML(
        f'<div style="font-family:sans-serif;font-size:15px;color:#1f2328;background:#f6f8fa;border-radius:8px;padding:10px;max-width:800px;margin:6px 0">'
        f"<b>학생 답:</b> {html.escape(answer)}<br><b>점수: {g.점수} / 3</b><br>"
        f'<div style="white-space:pre-wrap">{html.escape(g.근거)}</div><b>보완할 점:</b> {html.escape(g.보완할_점)}</div>'
    )

수강생 답을 아래에 적고 실행한다. 예시로 약한 답과 좋은 답을 하나씩 넣어 두었다.

In [ ]:
display(grade("모델이 느려서 그렇다."))
display(grade("모델은 호출할 도구와 인자만 텍스트로 만들 수 있고 실제 코드는 실행하지 못한다. "
              "그래서 앱이 tool_calls를 받아 함수를 실행하는데, 덕분에 어떤 도구를 실행할지 앱이 통제할 수 있다."))